# **Handling categorical data**

Real-world datasets often include **categorical data**, which must be converted into a format that numerical computing libraries can understand. This data is generally split into two distinct types:

---

#### **i. Ordinal Features**
These are categories with a **defined order or ranking**. Because there is a logical sequence, these values can be mapped to integers (e.g., $0, 1, 2$) while preserving their relative scale.
* **Example:** T-shirt size (**XL > L > M**).

#### **ii. Nominal Features**
These are categories that **do not imply any specific order**. One category is not "greater" or "smaller" than another; they are simply different labels.
* **Example:** T-shirt color (**Red, Blue, Green**).

---


#### **Comparison at a Glance**

| Feature Type | Has Logical Order? | Example | Numeric Strategy |
| :--- | :--- | :--- | :--- |
| **Ordinal** | Yes | Size (S, M, L) | Label Encoding (mapping to 1, 2, 3) |
| **Nominal** | No | Color (Red, Blue) | One-Hot Encoding (creating binary columns) |

---

**Documentation:** https://contrib.scikit-learn.org/category_encoders/

### **1. Categorical data encoding with pandas**

In [2]:
import pandas as pd

df = pd.DataFrame([
           ['green', 'M', 10.1, 'class2'],
           ['red', 'L', 13.5, 'class1'],
           ['blue', 'XL', 15.3, 'class2']])

df.columns = ['color', 'size', 'price', 'classlabel']

df

,color,size,price,classlabel
0,green,M,10.1,class2
1,red,L,13.5,class1
2,blue,XL,15.3,class2


### **2. Mapping Ordinal Features**

Since machine learning algorithms cannot inherently understand the "rank" within text-based categories, we must translate them into a numerical format that preserves their relative order.

---

### Key Takeaways

* **Integer Conversion:** To ensure a model understands that "Large" is greater than "Medium," categorical strings must be converted into **integers**.
* **The Manual Requirement:** Unlike some data tasks, there is no "magic button" to determine if *Large* should be a $2$ or a $10$. Because the order is based on human logic (domain knowledge), you must **manually define a mapping** (e.g., a Python dictionary).
* **Numerical Logic:** By assigning specific numbers, you establish the "distance" between categories. For example, assigning $M=1, L=2, XL=3$ tells the algorithm that the jump from $M$ to $L$ is mathematically equivalent to the jump from $L$ to $XL$.



---

### Example Mapping Logic

| String Label | Manual Integer Mapping | Mathematical Interpretation |
| :--- | :--- | :--- |
| **M** | 1 | Base Value |
| **L** | 2 | $M + 1$ |
| **XL** | 3 | $L + 1$ (or $M + 2$) |

In [3]:
size_mapping = {'XL': 3,
                'L' : 2,
                'M' : 1}

df['size'] = df['size'].map(size_mapping)
df

,color,size,price,classlabel
0,green,1,10.1,class2
1,red,2,13.5,class1
2,blue,3,15.3,class2


Reverse-mapping dictionary - Transform the integer values back to the original string representation.

In [4]:
inv_size_mapping = {v: k for k, v in size_mapping.items()}
df['size'].map(inv_size_mapping)

,size
0,M
1,L
2,XL


### **3. Encoding Class Labels**

Many machine learning libraries require **class labels** to be encoded as integer values. While some tools like `scikit-learn` can handle this internally, providing integer arrays is considered best practice to prevent technical glitches.

---

#### **3.1. Manual Mapping Approach**
Because class labels are not ordinal, you can assign any unique integer to a string label, typically starting from 0.

**Create a mapping dictionary:**

In [5]:
import numpy as np

class_mapping = {label: idx for idx, label in enumerate(np.unique(df['classlabel']))}
class_mapping

{'class1': 0, 'class2': 1}

**Apply and Reverse Mapping:**
* **Transform:** `df['classlabel'] = df['classlabel'].map(class_mapping)`
* **Inverse:** Create an inverse dictionary `{v: k for k, v in class_mapping.items()}` to map integers back to original strings.

---

#### **3.2. Using Scikit-Learn's LabelEncoder**
A more convenient way to achieve this is by using the `LabelEncoder` class from the `sklearn.preprocessing` module.

**Implementation:**



In [6]:
from sklearn.preprocessing import LabelEncoder

class_le = LabelEncoder()
y = class_le.fit_transform(df['classlabel'].values)
y

array([1, 0, 1])

**Reverting Labels:**
To convert integer labels back to their original string representation, use the `inverse_transform` method:

In [7]:
class_le.inverse_transform(y)

array(['class2', 'class1', 'class2'], dtype=object)

### **4. Performing one-hot encoding on nominal features**

When dealing with nominal features (categorical data with no inherent order, like color), simple integer encoding can mislead a classifier. Models may incorrectly assume an order (e.g., $red (2) > green (1) > blue (0)$), leading to suboptimal results.

In [9]:
X = df[['color', 'size', 'price']].values
color_le = LabelEncoder()
X[:, 0] = color_le.fit_transform(X[:, 0])
X

array([[1, 1, 10.1],
       [2, 2, 13.5],
       [0, 3, 15.3]], dtype=object)

#### **4.1. One-Hot Encoding**

The standard workaround is one-hot encoding, which creates a new binary "dummy" feature for each unique value in the nominal column.

**Using Scikit-Learn**

You can use OneHotEncoder or ColumnTransformer to selectively transform specific columns while leaving others untouched.

In [10]:
from sklearn.preprocessing import OneHotEncoder

X = df[['color', 'size', 'price']].values

color_ohe = OneHotEncoder()
color_ohe.fit_transform(X[:, 0].reshape(-1, 1)).toarray()

array([[0., 1., 0.],
       [0., 0., 1.],
       [1., 0., 0.]])

In [11]:
from sklearn.compose import ColumnTransformer

X = df[['color', 'size', 'price']].values

color_ohe = OneHotEncoder()
color_ohe.fit_transform(X[:, 0].reshape(-1, 1)).toarray()

c_transf = ColumnTransformer([
    ('onehot', color_ohe, [0]),      # Transform the first column
    ('nothing', 'passthrough', [1, 2]) # Leave other columns as-is
])

c_transf.fit_transform(X).astype(float)

array([[ 0. ,  1. ,  0. ,  1. , 10.1],
       [ 0. ,  0. ,  1. ,  2. , 13.5],
       [ 1. ,  0. ,  0. ,  3. , 15.3]])

**Using Pandas**

The get_dummies method is a convenient way to convert string columns in a DataFrame into dummy features automatically.

In [12]:
import pandas as pd
pd.get_dummies(df[['price', 'color', 'size']])

,price,size,color_blue,color_green,color_red
0,10.1,1,False,True,False
1,13.5,2,False,False,True
2,15.3,3,True,False,False


#### **4.2. Addressing Multicollinearity**

One-hot encoding introduces **multicollinearity**, which can cause issues for methods requiring matrix inversion. Highly correlated features lead to numerically unstable estimates.

**The Solution:** Remove one feature column from the encoded array. No information is lost because if all remaining dummy features are 0, it implies the dropped category must be the active one.

* **In Pandas:** `Use drop_first=True`.

* **In Scikit-Learn:** `Set drop='first'` and `categories='auto'` in the OneHotEncoder.

In [14]:
# Dropping the first column in Scikit-Learn
color_ohe = OneHotEncoder(categories='auto', drop='first')
c_transf = ColumnTransformer([
    ('onehot', color_ohe, [0]),
    ('nothing', 'passthrough', [1, 2])
])

c_transf.fit_transform(X).astype(float)

array([[ 1. ,  0. ,  1. , 10.1],
       [ 0. ,  1. ,  2. , 13.5],
       [ 0. ,  0. ,  3. , 15.3]])

### **5. Encoding ordinal features**

If the numerical differences between ordinal categories are unknown, or if the exact difference between two ordinal values isn't well-defined, you can use **threshold encoding** instead of assigning simple integer values (like 0, 1, 2).

This approach converts a single ordinal feature into multiple binary (0/1) features based on value thresholds.

---

### Threshold Encoding Example
Imagine a `size` feature with the ordinal values: `M`, `L`, and `XL`.

Instead of mapping them directly to integers, you can split this single feature into two new binary features:
1.  `x > M` (Is the size larger than Medium?)
2.  `x > L` (Is the size larger than Large?)

#### Implementation in Pandas
You can easily achieve this value-threshold approach using the `apply` method along with custom `lambda` expressions in pandas.

In [15]:
df = pd.DataFrame([['green', 'M', 10.1, 'class2'],
                   ['red', 'L', 13.5, 'class1'],
                   ['blue', 'XL', 15.3, 'class2']])

df.columns = ['color', 'size', 'price', 'classlabel']

df

,color,size,price,classlabel
0,green,M,10.1,class2
1,red,L,13.5,class1
2,blue,XL,15.3,class2


In [16]:
import pandas as pd

# Create a feature that is 1 if size is L or XL, else 0
df['x > M'] = df['size'].apply(lambda x: 1 if x in {'L', 'XL'} else 0)

# Create a feature that is 1 if size is XL, else 0
df['x > L'] = df['size'].apply(lambda x: 1 if x == 'XL' else 0)

In [17]:
del df['size']
df

,color,price,classlabel,x > M,x > L
0,green,10.1,class2,0,0
1,red,13.5,class1,1,0
2,blue,15.3,class2,1,1
